# Transfer Learning for Energy Time-Series Forecasting

## PV → Household Grid Import under Data Scarcity

This is the main research experiment of the project.

### Research question

> Can a Transformer pretrained on PV generation forecasting improve household
> grid-import forecasting when only limited target-task data are available?

**Notebook 01:** Data preparation  
↓  
**Notebook 02:** LSTM forecasting  
↓  
**Notebook 03:** Transformer forecasting  
↓  
**Notebook 04:** **Transfer learning**

### Experimental design

1. Pretrain a Transformer on PV generation forecasting.
2. Transfer its learned representation to grid-import forecasting.
3. Use only **10%, 25%, 50% and 100%** of the available grid-import training data.
4. Compare transfer learning against the same Transformer trained from scratch.
5. Keep validation and test periods fixed across all experiments.


## Transfer strategy

The PV source model's **input projection and Transformer encoder** are transferred
to the grid-import model.

The final prediction head is **reinitialized**, because PV generation and grid
import are different physical quantities.

The transferred model is then fine-tuned on the grid-import target task.

The key hypothesis is that useful temporal representations learned from the
related PV forecasting task may reduce the amount of target-task data required.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


In [ ]:
DATA_PATH = "/content/residential4_model_data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

data = df[
    ["timestamp", "pv_hourly", "grid_import_hourly"]
].copy()

print("Dataset shape:", data.shape)
print("Missing values:")
print(data.isna().sum())


## 1. Fixed chronological split

The same 70/15/15 chronological split used in Notebook 03 is retained.

The validation and test periods are identical for every data-scarcity experiment.


In [ ]:
n = len(data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data.iloc[:train_end].copy()
val = data.iloc[train_end:val_end].copy()
test = data.iloc[val_end:].copy()

print("Train:", len(train), train["timestamp"].min(), "→", train["timestamp"].max())
print("Validation:", len(val), val["timestamp"].min(), "→", val["timestamp"].max())
print("Test:", len(test), test["timestamp"].min(), "→", test["timestamp"].max())


In [ ]:
LOOKBACK = 24

def make_sequences(values, lookback=24):
    values = np.asarray(values)
    X, y = [], []

    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i])

    return np.asarray(X), np.asarray(y)


def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mean_target = np.mean(y_true)
    nrmse = rmse / mean_target if mean_target != 0 else np.nan

    return {"MAE": mae, "RMSE": rmse, "nRMSE": nrmse}


## 2. Transformer architecture

The architecture is kept identical to Notebook 03 so that the transfer-learning
experiment uses the same model family.

It contains a linear input projection, sinusoidal positional encoding, two
Transformer Encoder layers, mean pooling and a prediction head.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(
            position * div_term[:pe[:, 1::2].shape[1]]
        )

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TimeSeriesTransformer(nn.Module):
    def __init__(
        self,
        input_size=1,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_size, d_model)

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=LOOKBACK
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x).squeeze(-1)


# 3. Pretrain the source model on PV

The source task is PV generation forecasting.

The source model is trained using the full PV training period. Its validation
period is used only for model selection and is not used as target-task training
data.


In [ ]:
def prepare_source_data():
    values = data["pv_hourly"].values.astype(np.float32)

    scaler = StandardScaler()
    scaler.fit(values[:train_end].reshape(-1, 1))

    scaled = scaler.transform(
        values.reshape(-1, 1)
    ).flatten().astype(np.float32)

    X_train, y_train = make_sequences(
        scaled[:train_end],
        LOOKBACK
    )

    X_val, y_val = make_sequences(
        scaled[train_end-LOOKBACK:val_end],
        LOOKBACK
    )

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32)
        ),
        batch_size=64,
        shuffle=True
    )

    X_val = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    y_val = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)

    return train_loader, X_val, y_val, scaler


def pretrain_source_model(epochs=30, patience=5):
    train_loader, X_val, y_val, scaler = prepare_source_data()

    model = TimeSeriesTransformer().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )

    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            prediction = model(X_batch)
            loss = criterion(prediction, y_batch)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        with torch.no_grad():
            val_prediction = model(X_val)
            val_loss = criterion(
                val_prediction,
                y_val
            ).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    return model, history, scaler


In [ ]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pv_source_model, source_history, pv_source_scaler = pretrain_source_model()

print("\nPV source model pretrained.")


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(source_history["train_loss"], label="Training loss")
plt.plot(source_history["val_loss"], label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("PV Source-Task Pretraining")
plt.legend()

plt.tight_layout()
plt.show()


# 4. Target-task data scarcity

The target training set is restricted to **10%, 25%, 50% and 100%** of the
available chronological training period.

The validation and test sets never change.

For each fraction, two models are trained:

**Scratch:** Transformer initialized randomly.

**Transfer:** PV-pretrained input projection and encoder, with a new
grid-import prediction head.


In [ ]:
DATA_FRACTIONS = [0.10, 0.25, 0.50, 1.00]


def prepare_target_data(train_fraction):
    values = data["grid_import_hourly"].values.astype(np.float32)

    available_train_end = max(
        LOOKBACK + 1,
        int(train_end * train_fraction)
    )

    # Fit target scaling only on the target data available for training.
    scaler = StandardScaler()
    scaler.fit(
        values[:available_train_end].reshape(-1, 1)
    )

    scaled = scaler.transform(
        values.reshape(-1, 1)
    ).flatten().astype(np.float32)

    X_train, y_train = make_sequences(
        scaled[:available_train_end],
        LOOKBACK
    )

    X_val, y_val = make_sequences(
        scaled[train_end-LOOKBACK:val_end],
        LOOKBACK
    )

    X_test, y_test = make_sequences(
        scaled[val_end-LOOKBACK:],
        LOOKBACK
    )

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32)
        ),
        batch_size=64,
        shuffle=True
    )

    X_val = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    y_val = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)
    X_test = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)

    return (
        train_loader,
        X_val,
        y_val,
        X_test,
        y_test,
        scaler,
        available_train_end
    )


In [ ]:
def initialise_transfer_model(source_model):
    target_model = TimeSeriesTransformer().to(DEVICE)

    # Transfer learned temporal representation.
    target_model.input_projection.load_state_dict(
        copy.deepcopy(source_model.input_projection.state_dict())
    )

    target_model.encoder.load_state_dict(
        copy.deepcopy(source_model.encoder.state_dict())
    )

    # The target prediction head remains randomly initialized.
    return target_model


def train_target_model(
    model,
    train_loader,
    X_val,
    y_val,
    epochs=30,
    patience=5,
    learning_rate=1e-3
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            prediction = model(X_batch)
            loss = criterion(prediction, y_batch)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        with torch.no_grad():
            val_prediction = model(X_val)
            val_loss = criterion(
                val_prediction,
                y_val
            ).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    return model, best_val_loss, epoch + 1


def evaluate_model(model, X_test, y_test, scaler):
    model.eval()

    with torch.no_grad():
        prediction_scaled = model(X_test).cpu().numpy()

    y_true = scaler.inverse_transform(
        np.asarray(y_test).reshape(-1, 1)
    ).flatten()

    y_pred = scaler.inverse_transform(
        prediction_scaled.reshape(-1, 1)
    ).flatten()

    return y_true, y_pred


# 5. Run the transfer-learning experiment

For every target-data fraction, the scratch and transfer models use the
**same target observations, same validation set, same test set, architecture,
optimizer and early-stopping settings**.

The only difference is the initialization of the transfer model.


In [ ]:
results = []
predictions = {}

for fraction in DATA_FRACTIONS:

    print("=" * 70)
    print(f"Target training fraction: {fraction:.0%}")
    print("=" * 70)

    (
        train_loader,
        X_val,
        y_val,
        X_test,
        y_test,
        target_scaler,
        available_train_end
    ) = prepare_target_data(fraction)

    print(
        f"Target training observations used: {available_train_end:,}"
    )

    # -----------------------------
    # A. Transformer from scratch
    # -----------------------------
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    scratch_model = TimeSeriesTransformer().to(DEVICE)

    scratch_model, scratch_val_loss, scratch_epochs = train_target_model(
        scratch_model,
        train_loader,
        X_val,
        y_val
    )

    scratch_true, scratch_pred = evaluate_model(
        scratch_model,
        X_test,
        y_test,
        target_scaler
    )

    scratch_metrics = regression_metrics(
        scratch_true,
        scratch_pred
    )

    # -----------------------------
    # B. PV → grid transfer
    # -----------------------------
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    transfer_model = initialise_transfer_model(
        pv_source_model
    )

    transfer_model, transfer_val_loss, transfer_epochs = train_target_model(
        transfer_model,
        train_loader,
        X_val,
        y_val
    )

    transfer_true, transfer_pred = evaluate_model(
        transfer_model,
        X_test,
        y_test,
        target_scaler
    )

    transfer_metrics = regression_metrics(
        transfer_true,
        transfer_pred
    )

    results.append({
        "Target data": fraction,
        "Training observations": available_train_end,
        "Scratch MAE": scratch_metrics["MAE"],
        "Transfer MAE": transfer_metrics["MAE"],
        "Scratch RMSE": scratch_metrics["RMSE"],
        "Transfer RMSE": transfer_metrics["RMSE"],
        "Scratch nRMSE": scratch_metrics["nRMSE"],
        "Transfer nRMSE": transfer_metrics["nRMSE"],
        "MAE improvement (%)": (
            100
            * (scratch_metrics["MAE"] - transfer_metrics["MAE"])
            / scratch_metrics["MAE"]
        ),
        "RMSE improvement (%)": (
            100
            * (scratch_metrics["RMSE"] - transfer_metrics["RMSE"])
            / scratch_metrics["RMSE"]
        )
    })

    predictions[fraction] = {
        "true": transfer_true,
        "scratch": scratch_pred,
        "transfer": transfer_pred
    }

    print(
        f"Scratch MAE: {scratch_metrics['MAE']:.4f} | "
        f"Transfer MAE: {transfer_metrics['MAE']:.4f}"
    )

    print(
        f"Scratch RMSE: {scratch_metrics['RMSE']:.4f} | "
        f"Transfer RMSE: {transfer_metrics['RMSE']:.4f}"
    )

results_df = pd.DataFrame(results)
results_df


## 6. Main results

Positive improvement means that the PV-pretrained model performs better than
the Transformer trained from scratch using the same amount of target data.

The **10% and 25% cases** are particularly important because they represent
the strongest target-data scarcity.


In [ ]:
display(
    results_df[
        [
            "Target data",
            "Training observations",
            "Scratch MAE",
            "Transfer MAE",
            "Scratch RMSE",
            "Transfer RMSE",
            "MAE improvement (%)",
            "RMSE improvement (%)"
        ]
    ]
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    results_df["Target data"] * 100,
    results_df["Scratch MAE"],
    marker="o",
    label="Transformer from scratch"
)

plt.plot(
    results_df["Target data"] * 100,
    results_df["Transfer MAE"],
    marker="o",
    label="PV → Grid transfer"
)

plt.xlabel("Available grid-import training data (%)")
plt.ylabel("Test MAE (kWh)")
plt.title("Target Data Availability vs Forecasting Error")
plt.xticks([10, 25, 50, 100])
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    results_df["Target data"] * 100,
    results_df["MAE improvement (%)"],
    marker="o"
)

plt.axhline(0, linestyle="--")

plt.xlabel("Available grid-import training data (%)")
plt.ylabel("Transfer-learning MAE improvement (%)")
plt.title("Benefit of PV → Grid Transfer Learning")
plt.xticks([10, 25, 50, 100])

plt.tight_layout()
plt.show()


# 7. Example forecasts

The **25% target-data case** is shown as a representative data-scarcity
scenario.

The plot compares the actual grid-import series with the Transformer trained
from scratch and the PV-pretrained Transformer.


In [ ]:
fraction_to_plot = 0.25
example = predictions[fraction_to_plot]

plot_n = min(168, len(example["true"]))
test_timestamps = test["timestamp"].reset_index(drop=True)

plt.figure(figsize=(15, 5))

plt.plot(
    test_timestamps.iloc[:plot_n],
    example["true"][:plot_n],
    label="Actual",
    linewidth=1.2
)

plt.plot(
    test_timestamps.iloc[:plot_n],
    example["scratch"][:plot_n],
    label="Scratch Transformer",
    linewidth=1
)

plt.plot(
    test_timestamps.iloc[:plot_n],
    example["transfer"][:plot_n],
    label="PV → Grid Transfer",
    linewidth=1
)

plt.xlabel("Time")
plt.ylabel("Grid-import energy (kWh)")
plt.title("25% Target Data – Grid-Import Forecasting")
plt.legend()

plt.tight_layout()
plt.show()


# 8. Research interpretation

The main comparison is **transfer learning versus training from scratch at the
same target-data fraction**.

### If transfer learning improves at low data fractions

This supports the hypothesis that PV forecasting provides useful transferable
temporal representations for grid-import forecasting.

### If the benefit disappears as target data increase

This would suggest that transfer learning is most useful under data scarcity,
while a sufficiently large target dataset allows the scratch model to catch up.

### If transfer learning does not improve

That is also a valid research result. It would indicate that the selected
source and target tasks do not provide sufficiently useful transferable
representations under the current architecture and preprocessing.

The conclusion should therefore be based on the measured results rather than
assuming that transfer learning must outperform the baseline.


# Conclusions

This notebook completes the main technical progression:

**Data preparation**  
→ **LSTM forecasting**  
→ **Transformer forecasting**  
→ **Transfer learning under target-data scarcity**

It demonstrates:

- PyTorch Transformer modelling
- Source-task pretraining
- Cross-task model transfer
- Target-task fine-tuning
- Controlled 10/25/50/100% data-scarcity experiments
- Chronological evaluation
- Comparison against an identical model trained from scratch

### Final research question

> Can knowledge learned from PV generation forecasting be transferred to
> household grid-import forecasting, particularly when target-task data are
> limited?

The results above provide the evidence needed to answer this question.
